# SETUP FOR BETTER GPU

1) Install UCSD VPN: https://blink.ucsd.edu/technology/network/connections/off-campus/VPN/
2) ssh on local machine to ssh username@dsmlp-login.ucsd.edu where username is <username>@ucsd.edu
3) Run the following command on the ssh: launch.sh -c 8 -m 16 -g 1 -v a30
4) Open the link it provides
5) Make sure to **shut down** the pod when you're done from File > Shut Down. And then exit ssh from your terminal

6) If it says GPU already in use (1 of 1 allocated): run the following "kubectl get pods" and "kubectl delete pod <paste-pod-name-here>"


## Starter Code for Datahub

### IF THIS IS YOUR FIRST TIME RUNNING THIS ON DATA HUB
You need to run the cell immediately below so that you can install/setup the kernel. Its called "Python (cse151b)". You should **always** select this kernel in the top right of jupyter. 

### If you're coming back, make sure you re-run everything from the next cell (imports) and further down

The purpose of this starter code is so that you can basically duplicate this and run your experiments in a separate notebook.

# START HERE IF ITS YOUR FIRST TIME HERE

In [ ]:
# # =========================================================
# # DATAHUB SETUP & IMPORTS. (May take a while!) Comment me out after first run!
# # =========================================================

# # remove old kernel
# !jupyter kernelspec uninstall -y cse151b

# # Remove old .venv
# !rm -rf .venv

# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# print("Old environment and kernel destroyed.")

# # Create a virtual environment
# !~/.local/bin/uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm==0.19.1 tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!")

# START HERE IF YOU'VE ALREADY SETUP! 

# MAKE SURE TO RESTART KERNEL BEFORE CONTINUING!

## Library Setup

In [ ]:
!source ./.venv/bin/activate

import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
# from vllm import LLM, SamplingParams
from tqdm import tqdm
from datasets import load_dataset

## Config + Data Loading
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
 = 
GPU_ID      = "0" 
DATA_PATH   = "data/openr1_sample_stratified.jsonl"
OUTPUT_PATH = "results/sft_experiments.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

data = [json.loads(line) for line in open(DATA_PATH)]

# n_mcq  = sum(bool(d.get("options")) for d in data)
# n_free = sum(not d.get("options")   for d in data)
# print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# # Preview one MCQ and one free-form item
# mcq_sample  = next(d for d in data if d.get("options"))
# free_sample = next(d for d in data if not d.get("options"))

# print("\n── MCQ sample ──")
# print(json.dumps(mcq_sample, indent=2))
# print("\n── Free-form sample ──")
# print(json.dumps(free_sample, indent=2))

## Hyperparam Sandbox + LLM Setup

In [ ]:
# OLD HYPERPARAMETERS

# dtype="bfloat16", 
# quantization=None, 
# load_format="auto",
# enable_prefix_caching=True, 
# enforce_eager=False,        
# gpu_memory_utilization=0.90, 
# max_model_len=16384,
# trust_remote_code=True,
# max_num_seqs=256,
# max_num_batched_tokens=32768,
# max_tokens=16384, # CHANGED THIS
# temperature=0.6,
# top_p=0.95,
# top_k=20,
# presence_penalty=0.0,
# repetition_penalty=1.05,

from transformers import AutoModelForCausalLM, GenerationConfig
from trl import SFTTrainer, SFTConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM(MODEL_ID)
gen_config = GenerationConfig(
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    tokenizer=tokenizer
)
llm.generation_config = gen_config

trainer = SFTTrainer(
    model=llm,
    args=SFTConfig(
        max_length=16384
    ),
    tokenizer=tokenizer,
    train_dataset=data,
)
trainer.train()

print("Model loaded.")

In [ ]:
# SAVE MODEL

MODEL_DIR = "./try1"

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print(f"Model successfully saved to {MODEL_DIR}")

In [ ]:
# ACCESS MODEL

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
llm = AutoModelForCausalLM(MODEL_DIR)
gen_config = GenerationConfig(
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    tokenizer=tokenizer
)
llm.generation_config = gen_config

## Prompt Engineering Sandbox

In [7]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

# SYSTEM_PROMPT_MCQ = (
#     "You are an expert mathematician. "
#     "Read the problem and the answer choices below, then select the single best answer. "
#     "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
# )

# CHANGE SET #3:
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices carefully. "
    "Think through the problem step by step, then select the single best answer. "
    "At the very end of your response, output the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

## Inference (first 5 questions)

### 'responses' below only has the first 5 entries, ignore this and DO NOT USE IT. instead i have defined another list called responses_50, which is using the stratified training data that we will be using for this project. use responses_50 instead. i have commented it out below jic.

In [8]:
# #TODO: create smaller experiment set in data loading section and run inference on this!
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## dahi deepal addition - hyperparameter tuning

In [9]:
# pip install scikit-learn

In [10]:
import random
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [11]:
# getting n=50 stratified samples:

SUBSET_SIZE = 50
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

mcq_pool  = [d for d in data if d.get("options")]
free_pool = [d for d in data if not d.get("options")]

mcq_ratio = len(mcq_pool) / len(data)
n_mcq  = max(1, round(SUBSET_SIZE * mcq_ratio))
n_free = SUBSET_SIZE - n_mcq

experiment_data = (
    random.sample(mcq_pool,  n_mcq) +
    random.sample(free_pool, n_free)
)
random.shuffle(experiment_data)

print(f"Full dataset  : {len(data)} total  ({len(mcq_pool)} MCQ, {len(free_pool)} free-form)")
print(f"Subset        : {len(experiment_data)} total  ({n_mcq} MCQ, {n_free} free-form)")


Full dataset  : 1126 total  (375 MCQ, 751 free-form)
Subset        : 50 total  (17 MCQ, 33 free-form)


### this is where i have made responses_50:

In [ ]:
# Build prompts for first 50 entries
prompts = []
for item in experiment_data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)
# Added manual chunking because above line was causing EngineDead error
################################################################################
BATCH_SIZE = 8   # start small

all_outputs = []

for i in range(0, len(prompts), BATCH_SIZE):
    batch = prompts[i:i + BATCH_SIZE]

    print(f"Processing batch {i//BATCH_SIZE + 1}")

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    # 3. Generate the output
    # Note: It will automatically use the GenerationConfig you saved earlier, 
    # but you can override parameters here if needed.
    outputs = llm.generate(
        **inputs,
        max_new_tokens=16384,       # Limit the length of the generated response
        pad_token_id=tokenizer.eos_token_id
    )

    # 4. Decode and print the result
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response)

    all_outputs.extend(response)
################################################################################

responses_50 = [out.outputs[0].text.strip() for out in all_outputs]

# Preview first 3
for i in range(min(3, len(responses_50))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses_50[i][:400], "..." if len(responses_50[i]) > 400 else "")

Generating responses for 50 questions...
Processing batch 1


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing batch 2


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing batch 7


Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to figure out how long it takes for all three video editors—Level III, Level II, Level I—to work together on adding special effects. Hmm, this sounds like a work rate problem. Let me recall how those work. Usually, you find each person's rate of doing the ...

── Response 1 (id=1) ──
Okay, let's tackle this problem step by step. The question is asking whether each of the given functions is a power function. First, I need to remember what a power function is. From what I recall, a power function is a function of the form \( y = kx^n \), where \( k \) and \( n \) are constants, and \( n \) is a real number. The key here is that it's a single term with a variable raised to a cons ...

── Response 2 (id=2) ──
Okay, let's try to figure out this problem step by step. Hmm, so we have Kate and John working on a GRE math practi

In [13]:
# creating 4 metric formulas (accuracy, precision, recall, f1 score):
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

all_round_metrics = {}

def compute_metrics(results, label=""):
    y_true = [1] * len(results)                        # every question has a correct answer
    y_pred = [1 if r["correct"] else 0 for r in results]  # 1 if model got it right

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    mcq_res  = [r for r in results if r["is_mcq"]]
    free_res = [r for r in results if not r["is_mcq"]]

    def subset_scores(subset):
        yt = [1] * len(subset)
        yp = [1 if r["correct"] else 0 for r in subset]
        return (
            accuracy_score(yt, yp),
            precision_score(yt, yp, zero_division=0),
            recall_score(yt, yp, zero_division=0),
            f1_score(yt, yp, zero_division=0),
            sum(yp),
            len(yp)
        )

    print(f"\n{'='*50}")
    print(f"METRICS — {label}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Correct   : {sum(y_pred)} / {len(y_pred)}")

    for subset, name in [(mcq_res, "MCQ"), (free_res, "Free-form")]:
        if not subset:
            continue
        a, p, r, f, correct, total = subset_scores(subset)
        print(f"\n  [{name}]")
        print(f"    Accuracy  : {a:.4f}  ({a*100:.2f}%)")
        print(f"    Precision : {p:.4f}")
        print(f"    Recall    : {r:.4f}")
        print(f"    F1        : {f:.4f}")
        print(f"    Correct   : {correct} / {total}")

    print(f"{'='*50}")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

def record_round(round_name, results):
    metrics = compute_metrics(results, label=round_name)
    all_round_metrics[round_name] = metrics
    return metrics

def print_comparison():
    print(f"\n{'Round':<20} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("-" * 65)
    for name, m in all_round_metrics.items():
        print(f"{name:<20} {m['accuracy']:>9.4f}  {m['precision']:>9.4f}  {m['recall']:>9.4f}  {m['f1']:>9.4f}")

## Score Responses + Summary

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [14]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
# for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
for item, response in tqdm(zip(experiment_data, responses_50), total=len(data), desc="Scoring"): # CHANGED data TO experiment_data

    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")


mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

Scoring:   4%|▍         | 50/1126 [00:02<00:43, 24.99it/s]

Scoring complete. 50 results.
EVALUATION RESULTS
  MCQ        :   13 /   17  (76.47%)
  Free-form  :   17 /   33  (51.52%)
  Overall    :   30 /   50  (60.00%)


In [15]:
# computes and stores the metrics for this round
record_round("round1_baseline", results)


METRICS — round1_baseline
  Accuracy  : 0.6000  (60.00%)
  Precision : 1.0000
  Recall    : 0.6000
  F1 Score  : 0.7500
  Correct   : 30 / 50

  [MCQ]
    Accuracy  : 0.7647  (76.47%)
    Precision : 1.0000
    Recall    : 0.7647
    F1        : 0.8667
    Correct   : 13 / 17

  [Free-form]
    Accuracy  : 0.5152  (51.52%)
    Precision : 1.0000
    Recall    : 0.5152
    F1        : 0.6800
    Correct   : 17 / 33


{'accuracy': 0.6, 'precision': 1.0, 'recall': 0.6, 'f1': 0.75}

## Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [16]:
SAVE_EVAL = True   # Set to False when running on the private test set

# out_path = Path(OUTPUT_PATH)
out_path = Path("results/hyperparameter_change3.jsonl")
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 50 records to results/hyperparameter_change3.jsonl


In [21]:
experiment_data

[{'question': 'A Level III video editor takes $18$ hours to add special effects to a movie. A Level II video editor takes $26$ hours and a Level I video editor takes $34$ hours. Find how long it would take them to add the special effects if they all work together. Answer: [ANS] hours',
  'answer': ['8.10183'],
  'id': 910},
 {'question': 'Are the functions given below power functions? Answer T (true) or F (false).\n[ANS] 1. $y=3x^3+2x^2$ [ANS] 2. $y=2/(x^3)$ [ANS] 3. $y=x^3/2$ [ANS] 4. $y=14x^{12}$ [ANS] 5. $y=\\sqrt{4x^4}$ [ANS] 6. $y=12 \\cdot 14^x$',
  'answer': ['F', 'T', 'T', 'T', 'T', 'F'],
  'id': 308},
 {'question': 'It takes Kate k days to write a GRE math practice test. It takes John j days to write a GRE math practice test. If Kate and John work on a practice test in alternating 2-day shifts, it takes them 10 days when Kate starts and 10.5 days when John starts. How long would it take the two to complete a practice test if Kate and John worked simultaneously?',
  'options': 

In [22]:
# import json

# # Save experiment_data to a JSON file
# with open("data/experiment_data.json", "w", encoding="utf-8") as f:
#     json.dump(experiment_data, f, indent=2, ensure_ascii=False)

# print("Saved experiment_data to experiment_data.json")

Saved experiment_data to experiment_data.json
